In [1]:
import pandas as pd 
import os
import numpy as np
import datetime

In [2]:
# Make a list of all the csv file paths in the directories
data_loc = 'data/2014-2024_data/'
excel_file_names = [data_loc+file for file in os.listdir(data_loc) if '.xlsx' in file]


In [3]:
# creates a dictionary of DataFrames with key given by each sheet
list_of_dataframes = []
for file in excel_file_names:
    temp = pd.read_excel(file, sheet_name=None)
    list_of_dataframes += list(temp.values())

In [4]:
data_2014_2024 = pd.concat(list_of_dataframes, ignore_index=True)

It seems like at some point the ttc changed terminology, where :
- 'Min Gap' == 'Gap'
- 'Min Delay' == 'Delay'
- 'Bound' == 'Direction'
- 'Date' == 'Report Date' 
- 'Line' == 'Route'
- 'Incident' ~ 'Incident ID'


In order to verify these claims are true, let's make sure that there is 
never an instance where both columns are *not* NaN. 

In [5]:
possible_mergers = [('Min Gap', 'Gap'), ('Min Delay', 'Delay'), ('Bound', 'Direction'), ('Date', 'Report Date'), ('Line', 'Route'), ('Incident', 'Incident ID')]
for test1,test2 in possible_mergers:
    print(f'Number of colliding parameters when comparing {test1} with {test2} :')
    print(data_2014_2024[~data_2014_2024[test1].isna()  & ~data_2014_2024[test2].isna()].shape[0])

Number of colliding parameters when comparing Min Gap with Gap :
0
Number of colliding parameters when comparing Min Delay with Delay :
0
Number of colliding parameters when comparing Bound with Direction :
0
Number of colliding parameters when comparing Date with Report Date :
0
Number of colliding parameters when comparing Line with Route :
0
Number of colliding parameters when comparing Incident with Incident ID :
889


It looks like we can merge all but 'Incident' with 'Incident ID'. We will 
try to conform to the terminology in the 2025 data. 

In [ ]:
# We keep left tuple column and drop the right tuple column after merging
mergers = [('Min Gap', 'Gap'), ('Min Delay', 'Delay'), ('Bound', 'Direction'), ('Date', 'Report Date'), ('Line', 'Route')]
for keep, destroy in mergers:
    # Because we have 'inplace=True', rerunning the cell will cause an error.
    # Therefore, we add a 'try' conditional. 
    try:
        data_2014_2024[keep] = data_2014_2024[keep].combine_first(data_2014_2024[destroy])
        data_2014_2024.drop(columns=[destroy], inplace=True)
    except:
        print('Cell did not run. Did you already run this cell?')

Finally, we sort the dataframe by the Date and then the time, in lexicographic order. 

In [7]:
# Strangely, some of the Time data is stamped with date. 
# In particular, these times are *always* listed as midnight, 
# So I will elect to convert these to NAs.
data_2014_2024[data_2014_2024.Time.apply(lambda x : isinstance(x, datetime.datetime))]

,Time,Day,Location,Incident,Min Delay,Min Gap,Vehicle,Date,Line,Bound,Incident ID
2688,2016-03-10 00:00:00,Thursday,Queenway and Park lawn,Diversion,36.0,44.0,4181.0,2016-03-10,501.0,B/W,NaN
17249,2020-05-21 00:00:00,Thursday,Ferry Docks streetcar stop,Investigation,1.0,2.0,4459.0,2020-05-21,509.0,W/B,NaN
17285,2020-05-23 00:00:00,Saturday,Dundas West Station,Mechanical,15.0,20.0,4444.0,2020-05-23,505.0,E/B,NaN
42120,2017-05-27 00:00:00,Saturday,Broadview STN,General Delay,12.0,17.0,4190.0,2017-05-27,505.0,W/B,NaN
130965,2019-08-04 00:00:00,Monday,spadina station,Mechanical,6.0,13.0,4421.0,2019-08-05,510.0,S/B,NaN
132835,2019-10-14 00:00:00,Monday,Fleet Loop,General Delay,8.0,22.0,4416.0,2019-10-14,509.0,E/B,NaN


In [8]:
def get_time(x):
    if type(x) == datetime.datetime:
        return pd.NA
    else:
        return x
    
data_2014_2024.Time = data_2014_2024.Time.apply(get_time)

In [9]:
data_2014_2024.sort_values(by=['Date', 'Time'], inplace=True)

In [10]:
data_2014_2024.to_csv('data/2014-2024data.csv', index=False)